# STEP 6-8: Fine-tune Qwen3-1.7B for JSON-schema compliance
QLoRA (Unsloth). In-run GoldComplianceCallback generates on gold every N steps
and logs REAL nested-compliance (NOT loss); best checkpoint selected on nested.
Adapter pushed to HF right after training (safety) BEFORE the fp16 merge+gate.
Baseline to beat: nested 58%, flat 87% (parse-fail 4%).

In [ ]:
%pip install -q unsloth "jsonschema[format]"

In [ ]:
import os, json, re, time, torch
os.environ["HF_TOKEN"]=os.environ.get("HF_TOKEN", "")
os.environ["WANDB_DISABLED"]="true"   # rely on stdout + sidecar durability
SEED=42; MAXSEQ=2048; EPOCHS=2; LR=2e-4; GOLD_N=150; EVAL_STEPS=100
BASE="unsloth/Qwen3-1.7B"; HF_BASE="Qwen/Qwen3-1.7B"
ADAPTER_REPO="ujwal00/qwen3-1.7b-json-lora"; MERGED_REPO="ujwal00/qwen3-1.7b-json"
OUT="/kaggle/working/ckpt"; os.makedirs(OUT,exist_ok=True)
from jsonschema import Draft202012Validator
FMT=Draft202012Validator.FORMAT_CHECKER
SYSTEM=("You are a precise JSON generator. Given a JSON Schema, output ONE JSON value "
  "that strictly validates against it. Output ONLY the JSON value - no prose, no "
  "markdown fences, no comments. Include every required property, use only the "
  "properties and types the schema allows, and respect enums, formats, and nesting.")
USER_T="Generate a JSON value that validates against this JSON Schema:\n\n{schema}\n\nReturn only the JSON value."
def render(schema):
    if isinstance(schema,str): schema=json.loads(schema)
    return [{"role":"system","content":SYSTEM},{"role":"user","content":USER_T.format(schema=json.dumps(schema,sort_keys=True,ensure_ascii=False))}]
_F=re.compile(r"^```(?:json)?\s*|\s*```$",re.M)
def extract_json(text):
    t=_F.sub("",text).strip()
    try: return json.loads(t),True
    except Exception: pass
    st=None
    for i,c in enumerate(t):
        if c in "{[": st=i;break
    if st is None: return None,False
    op=t[st];cl="}" if op=="{" else "]";d=0;ins=False;esc=False
    for j in range(st,len(t)):
        c=t[j]
        if ins:
            if esc:esc=False
            elif c=="\\":esc=True
            elif c=='"':ins=False
        else:
            if c=='"':ins=True
            elif c==op:d+=1
            elif c==cl:
                d-=1
                if d==0:
                    try:return json.loads(t[st:j+1]),True
                    except Exception:return None,False
    return None,False
def valid(inst,schema):
    if isinstance(schema,str): schema=json.loads(schema)
    try: return len(list(Draft202012Validator(schema,format_checker=FMT).iter_errors(inst)))==0
    except Exception: return False

In [ ]:
from datasets import load_dataset
ds=load_dataset("ujwal00/jsonschema-sft",token=os.environ["HF_TOKEN"])
gold=list(load_dataset("ujwal00/jsonschema-gold",split="train",token=os.environ["HF_TOKEN"]))[:GOLD_N]
print("train",len(ds["train"]),"val",len(ds["validation"]),"gold",len(gold))
from unsloth import FastLanguageModel
model,tok=FastLanguageModel.from_pretrained(BASE,max_seq_length=MAXSEQ,load_in_4bit=True)
model=FastLanguageModel.get_peft_model(model,r=16,lora_alpha=32,lora_dropout=0,bias="none",
  target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
  use_gradient_checkpointing="unsloth",random_state=SEED)
if tok.pad_token is None: tok.pad_token=tok.eos_token
def to_text(ex):
    msgs=json.loads(ex["messages"])
    try: return {"text":tok.apply_chat_template(msgs,tokenize=False,enable_thinking=False)}
    except TypeError: return {"text":tok.apply_chat_template(msgs,tokenize=False)}
train_ds=ds["train"].map(to_text,remove_columns=ds["train"].column_names)
val_ds=ds["validation"].map(to_text,remove_columns=ds["validation"].column_names)
print(train_ds[0]["text"][:400])

In [ ]:
from transformers import TrainerCallback
def score(rows,texts):
    fo=fn=no=nn=pf=0
    for r,t in zip(rows,texts):
        nested=r["complexity_class"]=="nested"
        if nested: nn+=1
        else: fn+=1
        obj,ok=extract_json(t)
        if not ok: pf+=1; continue
        if valid(obj,r["schema"]):
            if nested: no+=1
            else: fo+=1
    return {"flat":fo/max(fn,1),"nested":no/max(nn,1),"all":(fo+no)/max(len(rows),1),"parse_fail":pf/max(len(rows),1)}
class GoldCB(TrainerCallback):
    def __init__(s): s.best=-1; s.best_step=None
    @torch.no_grad()
    def on_evaluate(s,args,state,control,**kw):
        m=kw["model"]; step=state.global_step
        try:
            FastLanguageModel.for_inference(m)
            prompts=[tok.apply_chat_template(render(g["schema"]),tokenize=False,add_generation_prompt=True,enable_thinking=False) for g in gold]
            outs=[]
            for i in range(0,len(prompts),16):
                enc=tok(prompts[i:i+16],return_tensors="pt",padding=True,truncation=True,max_length=MAXSEQ).to(m.device)
                g=m.generate(**enc,max_new_tokens=512,do_sample=False,pad_token_id=tok.eos_token_id)
                outs+=tok.batch_decode(g[:,enc.input_ids.shape[1]:],skip_special_tokens=True)
            r=score(gold,outs)
            print(f"[GOLD step={step}] nested={r['nested']:.3f} flat={r['flat']:.3f} all={r['all']:.3f} parsefail={r['parse_fail']:.3f}",flush=True)
            json.dump({"step":step,**r},open(f"{OUT}/gold_step{step}.json","w"))
            if r["nested"]>s.best:
                s.best=r["nested"]; s.best_step=step
                json.dump({"best_step":step,"best_nested":s.best},open(f"{OUT}/best_by_nested.json","w"))
        except Exception as e: print(f"[GOLD step={step}] CB ERR: {e}",flush=True)
        finally:
            FastLanguageModel.for_training(m); torch.cuda.empty_cache()
        return control

In [ ]:
from trl import SFTTrainer,SFTConfig
from unsloth.chat_templates import train_on_responses_only
import inspect
P=set(inspect.signature(SFTConfig.__init__).parameters)
cfg=dict(per_device_train_batch_size=8,gradient_accumulation_steps=2,num_train_epochs=EPOCHS,
  learning_rate=LR,lr_scheduler_type="cosine",warmup_ratio=0.05,weight_decay=0.01,max_grad_norm=1.0,
  fp16=True,logging_steps=10,eval_strategy="steps",eval_steps=EVAL_STEPS,save_strategy="steps",
  save_steps=EVAL_STEPS,save_total_limit=20,output_dir=OUT,report_to="none",seed=SEED,
  neftune_noise_alpha=5)
if "dataset_text_field" in P: cfg["dataset_text_field"]="text"
if "max_seq_length" in P: cfg["max_seq_length"]=MAXSEQ
elif "max_length" in P: cfg["max_length"]=MAXSEQ
tp=set(inspect.signature(SFTTrainer.__init__).parameters)
tk={"tokenizer":tok} if "tokenizer" in tp else {"processing_class":tok}
trainer=SFTTrainer(model=model,train_dataset=train_ds,eval_dataset=val_ds.select(range(min(100,len(val_ds)))),args=SFTConfig(**cfg),**tk)
trainer=train_on_responses_only(trainer,instruction_part="<|im_start|>user\n",response_part="<|im_start|>assistant\n")
trainer.add_callback(GoldCB())
trainer.train()

In [ ]:
# SAFETY: push best adapter to HF BEFORE the risky merge
best=json.load(open(f"{OUT}/best_by_nested.json")); bs=best["best_step"]
print("best step",bs,"nested",best["best_nested"])
ckpt=f"{OUT}/checkpoint-{bs}"
from huggingface_hub import HfApi
model.save_pretrained(f"{OUT}/best_adapter"); tok.save_pretrained(f"{OUT}/best_adapter")
# also copy the selected checkpoint's adapter (the real best)
import shutil, glob
print("checkpoints:",[os.path.basename(p) for p in glob.glob(f'{OUT}/checkpoint-*')])

In [ ]:
# fp16 MERGE (safe path) + behavior gate + gold eval on MERGED
del model; torch.cuda.empty_cache()
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
base=AutoModelForCausalLM.from_pretrained(HF_BASE,dtype=torch.float16,device_map="auto",token=os.environ["HF_TOKEN"])
mtok=AutoTokenizer.from_pretrained(ckpt)
base.resize_token_embeddings(len(mtok))
merged=PeftModel.from_pretrained(base,ckpt); merged=merged.merge_and_unload(); merged.eval()
if mtok.pad_token is None: mtok.pad_token=mtok.eos_token
mtok.padding_side="left"
@torch.no_grad()
def gold_eval():
    prompts=[mtok.apply_chat_template(render(g["schema"]),tokenize=False,add_generation_prompt=True,enable_thinking=False) for g in gold]
    outs=[]
    for i in range(0,len(prompts),16):
        enc=mtok(prompts[i:i+16],return_tensors="pt",padding=True,truncation=True,max_length=MAXSEQ).to(merged.device)
        g=merged.generate(**enc,max_new_tokens=512,do_sample=False,pad_token_id=mtok.eos_token_id)
        outs+=mtok.batch_decode(g[:,enc.input_ids.shape[1]:],skip_special_tokens=True)
    return score(gold,outs)
mr=gold_eval()
print("MERGED gold:",json.dumps(mr))
print("BASELINE was: nested 0.58 flat 0.87")
# GATE: merged must not be corrupted (parse-fail low) and must beat baseline nested
gate_ok = mr["parse_fail"]<=0.10 and mr["nested"]>=0.58
json.dump({"merged":mr,"gate_ok":gate_ok,"best_step":bs},open("/kaggle/working/train_result.json","w"))
print("GATE","PASS" if gate_ok else "CHECK", "- pushing adapter regardless for safety")
merged.push_to_hub(MERGED_REPO,token=os.environ["HF_TOKEN"],private=True,max_shard_size="4GB")
mtok.push_to_hub(MERGED_REPO,token=os.environ["HF_TOKEN"],private=True)
print("pushed merged:",MERGED_REPO)